# Workshop — 3. Fine-Tuning SmolVLA on SageMaker

Notebook 1 collected successful SO100 simulation demonstrations. Notebook 2 established the zero-shot failure caused by domain shift. This notebook launches a SageMaker Training Job that updates the same SmolVLA checkpoint on the collected dataset.

The design follows one separation of concerns:

- the SageMaker PyTorch DLC provides PyTorch, CUDA, and the training toolkit;
- `ModelTrainer.SourceCode` uploads the training script and installs its requirements at job startup;
- the dataset and YAML configuration arrive through input channels;
- `scripts/train.py` contains no SageMaker-specific training logic.

## 1. SageMaker Session

This notebook uses the SageMaker Python SDK V3, matching the other training examples.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"SageMaker role ARN: {role}")
print(f"SageMaker bucket: {bucket_name}")
print(f"SageMaker region: {sess.boto_region_name}")

## 2. Upload the Dataset and Training Configuration

The complete LeRobot dataset tree must retain its `meta/`, `data/`, and `videos/` layout. SageMaker downloads the S3 prefix into `/opt/ml/input/data/train`. The training configuration is a separate `config` channel.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
DATASET_DIR = ROOT / "datasets" / "so100_sim_pickplace"
SCRIPTS_DIR = ROOT / "scripts"

info_path = DATASET_DIR / "meta" / "info.json"
if not info_path.is_file():
    raise FileNotFoundError(info_path)
info = json.loads(info_path.read_text())
assert info["robot_type"] == "so100"
assert info["features"]["observation.state"]["shape"] == [6]
assert info["features"]["action"]["shape"] == [6]
assert {
    key for key in info["features"] if key.startswith("observation.images.")
} == {"observation.images.top", "observation.images.wrist"}

print({
    "episodes": info["total_episodes"],
    "frames": info["total_frames"],
    "fps": info["fps"],
    "dataset": str(DATASET_DIR),
})

### Create `args.yaml`

The configuration is generated by the notebook, uploaded through the `config` channel, and removed locally after a successful upload. This keeps the job parameters visible and editable without baking them into the training image.

In [ ]:
import yaml

model_id = "lucarrr/smolvla_so100_pickplace_finetuned_v2"
model_revision = "4fde42badae91b5c88bfe1a399a3f4b994fc0698"

training_args = {
    "model": {
        "id": model_id,
        "revision": model_revision,
        "freeze_vision_encoder": True,
        "train_expert_only": True,
        "train_state_proj": True,
        "camera_mapping": {
            "observation.images.wrist": "observation.images.camera1",
            "observation.images.top": "observation.images.camera2",
        },
    },
    "dataset": {
        "repo_id": "local/so100_sim_pickplace",
        "video_backend": "pyav",
        "return_uint8": True,
        "eval_split": 0.125,
    },
    "training": {
        "use_amp": False,
        "job_name": "smolvla-so100-sim",
        "steps": 1000,
        "batch_size": 4,
        "num_workers": "auto",
        "prefetch_factor": 2,
        "persistent_workers": True,
        "learning_rate": 1e-4,
        "warmup_steps": 100,
        "decay_steps": 1000,
        "decay_lr": 2.5e-6,
        "log_freq": 10,
        "eval_steps": 250,
        "max_eval_samples": 128,
        "save_checkpoint": True,
        "save_freq": 500,
        "seed": 42
    },
    "tracking": {"wandb": False},
    "paths": {
        "dataset_dir": "/opt/ml/input/data/train",
        "work_dir": "/opt/ml/checkpoints/smolvla-run",
        "model_dir": "/opt/ml/model/smolvla",
    },
}

TRAINING_CONFIG = ROOT / "args.yaml"
TRAINING_CONFIG.write_text(
    yaml.safe_dump(training_args, sort_keys=False),
    encoding="utf-8",
)
print(TRAINING_CONFIG.read_text())

In [ ]:
if default_prefix:
    input_path = f"{default_prefix}/physical-ai/smolvla-so100"
else:
    input_path = "physical-ai/smolvla-so100"

dataset_key_prefix = f"{input_path}/dataset"
config_key = f"{input_path}/config/args.yaml"

for local_file in sorted(DATASET_DIR.rglob("*")):
    if local_file.is_file():
        relative = local_file.relative_to(DATASET_DIR).as_posix()
        s3_client.upload_file(
            str(local_file),
            bucket_name,
            f"{dataset_key_prefix}/{relative}",
        )

s3_client.upload_file(str(TRAINING_CONFIG), bucket_name, config_key)
TRAINING_CONFIG.unlink()

train_dataset_s3_path = f"s3://{bucket_name}/{dataset_key_prefix}/"
train_config_s3_path = f"s3://{bucket_name}/{config_key}"

print("Dataset:", train_dataset_s3_path)
print("Config:", train_config_s3_path)

## 3. Use the Native SageMaker PyTorch Container

The default path does not build or maintain a custom image. `image_uris.retrieve` selects the SageMaker PyTorch training DLC compatible with the chosen instance type.

In [ ]:
from sagemaker.core import image_uris

instance_type = "ml.g5.2xlarge"
instance_count = 1

image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sess.boto_session.region_name,
    version="2.8.0",
    instance_type=instance_type,
    image_scope="training",
)

print(image_uri)

## 4. Configure the Training Job

`SourceCode` packages `scripts/train.py` independently from the container. `requirements.txt` is installed when the job starts. `Torchrun()` provides the same distributed environment that the LeRobot/Accelerate loop uses locally.

In [ ]:
from sagemaker.train.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.distributed import Torchrun
from sagemaker.train.model_trainer import ModelTrainer

source_code = SourceCode(
    source_dir=str(SCRIPTS_DIR),
    requirements="requirements.txt",
    entry_script="train.py",
)

compute_configs = Compute(
    instance_type=instance_type,
    instance_count=instance_count,
    volume_size_in_gb=100,
    keep_alive_period_in_seconds=1800,
)

job_name = "train-smolvla-so100-sim"
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    distributed=Torchrun(),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=4 * 60 * 60),
    hyperparameters={
        "config": "/opt/ml/input/data/config/args.yaml",
        "device": "cuda",
    },
    output_data_config=OutputDataConfig(
        s3_output_path=output_path,
        compression_type="NONE",
    ),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoint",
        local_path="/opt/ml/checkpoints",
    ),
)

print(model_trainer)

In [ ]:
from sagemaker.train.configs import InputData

train_input = InputData(
    channel_name="train",
    data_source=train_dataset_s3_path,
)
config_input = InputData(
    channel_name="config",
    data_source=train_config_s3_path,
)

data = [train_input, config_input]
data

## 5. Launch

The call returns immediately because `wait=False`. SageMaker streams checkpoints to the configured checkpoint S3 prefix and uploads the exported model from `/opt/ml/model`.

In [ ]:
training_job = model_trainer.train(input_data_config=data, wait=False)
training_job

## Expected Artifact

The job exports a complete SmolVLA checkpoint with:

- simulation-native `observation.images.wrist` and `observation.images.top` inputs;
- 6D SO100 state and action features;
- normalization statistics computed from the simulation dataset in radians;
- a `training_manifest.json` recording the source checkpoint and dataset fingerprint.

Training loss is not the final workshop metric. Download the completed model artifact and repeat Notebook 2's rollout using the same video and `placed_in_box` check.